# 03. SOVIGAZ DEMAND & REVENUE FORECASTING (PHASE 3)
**Project**: SOVIGAZ Machine Learning System  
**Author**: Senior Machine Learning Engineer & Data Scientist  
**Mục tiêu**:
1. Dự báo Tổng Doanh thu toàn công ty cấp Tuần & cấp Tháng (Weekly & Monthly Revenue Forecasting).
2. Dự báo Nhu cầu Sản lượng Khí Chủ lực (`MainQuantity` cho Oxy lỏng `OXL-SX`, Nitơ lỏng `NTL-SX`, Oxy khí chai 6m³ `OXK6`).
3. Đánh giá Baselines (Naive, Seasonal Naive, Moving Average) đối chiếu với các mô hình Machine Learning (LightGBM, XGBoost, CatBoost) và SARIMAX.
4. Kiểm định Chronological Split không rò rỉ dữ liệu và tích hợp đặc trưng ngoại sinh Tết Nguyên Đán.
5. Xuất báo cáo Power BI chuẩn hóa (`forecast_powerbi.csv`).

In [ ]:
import os
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
sys.path.insert(0, str(project_root))

from config.config import PROCESSED_DATA_DIR, OUTPUTS_DIR, MODELS_DIR
print('Project Root:', project_root)
print('Processed Data Dir:', PROCESSED_DATA_DIR)
print('Outputs Dir:', OUTPUTS_DIR)


## 1. Bảng So Sánh Hiệu Năng Mô Hình (Model Benchmarking across Metrics)

In [ ]:
with open(MODELS_DIR / 'metrics.json', 'r', encoding='utf-8') as f:
    metrics_data = json.load(f)

print('=== 1.1 WEEKLY REVENUE METRICS ===')
df_m_w = pd.DataFrame(metrics_data['Weekly_Revenue']).sort_values(by='WAPE')
display(df_m_w)

print('=== 1.2 MONTHLY REVENUE METRICS ===')
df_m_m = pd.DataFrame(metrics_data['Monthly_Revenue']).sort_values(by='WAPE')
display(df_m_m)

print('=== 1.3 PRODUCT DEMAND METRICS ===')
df_m_p = pd.DataFrame(metrics_data['Product_Demand']).sort_values(by=['Product', 'WAPE'])
display(df_m_p)


## 2. Kết quả Dự báo Doanh thu Toàn Công ty cấp Tuần (Weekly Revenue: Ground Truth vs Model Forecast)

In [ ]:
df_w_forecast = pd.read_csv(OUTPUTS_DIR / 'forecasting' / 'weekly_revenue_forecast.csv')
display(df_w_forecast.head(10))

# Visualize Actual vs Forecast
fig, ax = plt.subplots(figsize=(11, 4.5), dpi=150)
ax.plot(pd.to_datetime(df_w_forecast['Date']), df_w_forecast['Actual'] / 1e9, label='Actual Revenue (Ground Truth)', color='#1F497D', marker='o', linewidth=2)
ax.plot(pd.to_datetime(df_w_forecast['Date']), df_w_forecast['Forecast'] / 1e9, label=f"Forecast ({df_w_forecast['Model'].iloc[0]})", color='#D9534F', linestyle='--', marker='s', linewidth=2)
ax.set_title('Weekly Revenue: Ground Truth vs CatBoost Forecast (2026 Test Horizon - 23 Complete Weeks)', pad=12)
ax.set_ylabel('Doanh thu thuần (Tỷ VND)')
ax.set_xlabel('Tuần giao dịch')
ax.legend()
plt.tight_layout()
plt.show()


## 3. Phân tích Mùa Vụ Tết & Doanh thu Cấp Tháng (Monthly Revenue)

In [ ]:
df_m_forecast = pd.read_csv(OUTPUTS_DIR / 'forecasting' / 'monthly_revenue_forecast.csv')
display(df_m_forecast)

fig, ax = plt.subplots(figsize=(9, 4), dpi=150)
ax.plot(pd.to_datetime(df_m_forecast['Date']), df_m_forecast['Actual'] / 1e9, label='Actual Revenue', color='#1F497D', marker='o', linewidth=2)
ax.plot(pd.to_datetime(df_m_forecast['Date']), df_m_forecast['Forecast'] / 1e9, label=f"Forecast ({df_m_forecast['Model'].iloc[0]})", color='#2E7D32', linestyle='--', marker='^', linewidth=2)
ax.set_title('Monthly Revenue Forecast vs Actual (Jan 2026 - May 2026 Complete Months)', pad=12)
ax.set_ylabel('Doanh thu thuần (Tỷ VND)')
ax.legend()
plt.tight_layout()
plt.show()


## 4. Dự báo Nhu cầu Sản lượng Khí Chủ lực (MainQuantity: OXL-SX, NTL-SX, OXK6)

In [ ]:
df_prod = pd.read_csv(OUTPUTS_DIR / 'forecasting' / 'product_demand_forecast.csv')
p_summary = df_prod.groupby('Product').agg(
    Test_Periods=('Date', 'count'),
    Total_Actual=('Actual', 'sum'),
    Total_Forecast=('Forecast', 'sum'),
    Best_Model=('Model', 'first')
).reset_index()
p_summary['Volume_Error_%'] = round(abs(p_summary['Total_Actual'] - p_summary['Total_Forecast']) / p_summary['Total_Actual'] * 100, 2)
display(p_summary)


## 5. Dữ liệu Xuất Chuẩn Hóa cho Power BI (forecast_powerbi.csv)

In [ ]:
df_pbi = pd.read_csv(OUTPUTS_DIR / 'forecasting' / 'forecast_powerbi.csv')
print(f'Total Power BI forecast records: {len(df_pbi)}')
display(df_pbi.head(15))
